# ASVspoof 2021 LA – dataset check

This notebook checks whether the ASVspoof 2021 LA files on disk match what is written in the official documentation: `README.LA.txt`, `keys/LA/README.txt`, the eval-package README and the main paper (Liu et al., 2023, arXiv version 3).

**How to use it**

- Run the cells from top to bottom.
- The notebook only **reads** the data – nothing in `datasets/` is changed.
- Every check prints one line:
  - `[OK]` – the files match the documentation,
  - `[DIFF]` – the files do not match the documentation,
  - `[INFO]` – an observation, there is nothing to compare with.

**Output files**

In `outputs/02_asvspoof_2021_la_dataset_exploration_output/` (not committed to git):

- `audio_info.csv` – properties of every audio file (step 8a),
- `checks.csv` – table of all checks (step 9).

In `results/02_asvspoof_2021_la/02_asvspoof_2021_la_dataset_exploration_output/` (committed to git):

- `dataset_exploration_report.md` – report with the results of all steps (step 10).

### Setup – paths and settings

**What it does:** Loads the libraries, finds the project folder and sets the paths to the dataset (`datasets/02_asvspoof_2021_la`), to the outputs folder for working files and to the results folder for the report. It also defines the names of the three subsets (progress, eval, hidden), of the seven codecs and of the columns of the label file.

**Why:** All the next cells use these paths and names. The paths are relative to the project folder, so the notebook works on any computer without changes.

In [ ]:
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import soundfile as sf
from IPython.display import display


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "pyproject.toml").exists():
            return path
    raise FileNotFoundError("pyproject.toml not found")


REPO_ROOT = find_repo_root(Path.cwd())
DATA_DIR = REPO_ROOT / "datasets" / "02_asvspoof_2021_la"
EVAL_DIR = DATA_DIR / "ASVspoof2021_LA_eval"
FLAC_DIR = EVAL_DIR / "flac"
KEYS_DIR = DATA_DIR / "keys" / "LA"
NOTEBOOK_NAME = "02_asvspoof_2021_la_dataset_exploration"
OUTPUTS_DIR = REPO_ROOT / "outputs" / f"{NOTEBOOK_NAME}_output"
RESULTS_DIR = REPO_ROOT / "results" / "02_asvspoof_2021_la" / f"{NOTEBOOK_NAME}_output"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

METADATA_FILE = KEYS_DIR / "CM" / "trial_metadata.txt"
TRIAL_LIST_FILE = EVAL_DIR / "ASVspoof2021.LA.cm.eval.trl.txt"
COLUMNS = ["speaker_id", "file_name", "codec", "transmission", "attack", "key", "trim", "subset"]
SUBSETS = ["progress", "eval", "hidden"]
# values of the codec column in the order of the conditions LA-C1 ... LA-C7 (main paper, Table II)
CODECS = ["none", "alaw", "pstn", "g722", "ulaw", "gsm", "opus"]
EVAL_PACKAGE = "eval-package README (web)"

print("Data folder:", DATA_DIR.relative_to(REPO_ROOT), "| exists:", DATA_DIR.exists())

### Check list – helper function

**What it does:** Defines `add_check`, a small function that compares an **expected** value (from the documentation) with a **measured** value (from the files on disk). It prints `[OK]` if they match, `[DIFF]` if they do not, and `[INFO]` if there is nothing to compare with (only an observation). Every result is saved in the list `checks`.

**Why:** All results are collected in one place, so at the end they can be shown in one table and written to the report.

In [ ]:
checks = []


def add_check(check: str, source: str, expected, measured) -> None:
    ok = None if expected is None else expected == measured
    checks.append(
        {
            "check": check,
            "source": source,
            "expected": expected,
            "measured": measured,
            "ok": ok,
        }
    )
    status = "INFO" if ok is None else ("OK" if ok else "DIFF")
    print(f"[{status}] {check}: expected={expected!r}, measured={measured!r}")

### Step 1 – Folder structure

**What it does:** Lists the files inside `ASVspoof2021_LA_eval/` and compares them with the list in `README.LA.txt` (section 1). It also checks that the keys package contains the two label files described in `keys/LA/README.txt`.

**Why:** Confirms that the audio package and the keys package were downloaded and unpacked completely.

In [ ]:
found_top = sorted(p.name for p in DATA_DIR.iterdir() if not p.name.startswith("."))
print("Data folder contains:", found_top)

expected_eval_items = {"ASVspoof2021.LA.cm.eval.trl.txt", "flac", "README.LA.txt", "LICENSE.txt"}
found_eval_items = {p.name for p in EVAL_DIR.iterdir() if not p.name.startswith(".")}
add_check(
    "ASVspoof2021_LA_eval folder contents",
    "README.LA.txt, section 1",
    expected_eval_items,
    found_eval_items,
)

key_files = ["CM/trial_metadata.txt", "ASV/trial_metadata.txt"]
add_check(
    "keys/LA: CM and ASV trial_metadata.txt exist",
    "keys/LA/README.txt",
    True,
    all((KEYS_DIR / name).exists() for name in key_files),
)

### Step 2 – Load the label files

**What it does:** Reads the label file `keys/LA/CM/trial_metadata.txt` into one table. Each row is one recording: speaker, file name, codec, transmission, attack, real (`bonafide`) or fake (`spoof`), trimming and subset. It also reads the list of evaluation files `ASVspoof2021.LA.cm.eval.trl.txt` from the audio package.

**Why:** All the next checks use these tables. In ASVspoof 2021 the labels are not in the audio package – they come in the separate keys package.

In [ ]:
cm = pd.read_csv(
    METADATA_FILE,
    sep=" ",
    header=None,
    names=COLUMNS,
    dtype=str,
    keep_default_na=False,
)
trials = pd.read_csv(
    TRIAL_LIST_FILE,
    sep=" ",
    header=None,
    names=["file_name"],
    dtype=str,
    keep_default_na=False,
)
print("Rows in trial_metadata.txt:", len(cm))
print("Rows in the trial list:", len(trials))
display(cm)

### Step 3 – Format of the label files

**What it does:** Checks that every line of `trial_metadata.txt` has 8 columns, which values appear in the columns label, attack, subset and trimming, and that no file name appears twice.

**Why:** If the label files were broken, every later result would be wrong. The expected values for key, subset and trimming come from the eval-package README (link 5 in section 3 of `docs/02_asvspoof_2021_la.md`). For trimming the README gives `notrim` / `trim`, but the file uses `only_speech` – this known difference is kept as a DIFF.

In [ ]:
def columns_per_line(path: Path) -> set:
    with open(path, encoding="utf-8") as f:
        return {len(line.split()) for line in f if line.strip()}


add_check(
    "trial_metadata.txt: columns per line", EVAL_PACKAGE, {8}, columns_per_line(METADATA_FILE)
)
add_check("Values of key", EVAL_PACKAGE, {"bonafide", "spoof"}, set(cm["key"]))
add_check(
    "attack for spoof",
    "main paper, p. 3, section II-A",
    {f"A{i:02d}" for i in range(7, 20)},
    set(cm.loc[cm["key"] == "spoof", "attack"]),
)
add_check("Values of subset", EVAL_PACKAGE, set(SUBSETS), set(cm["subset"]))
add_check("Values of trim", EVAL_PACKAGE, {"notrim", "trim"}, set(cm["trim"]))
add_check("Duplicated file names", "-", 0, int(cm["file_name"].duplicated().sum()))

### Step 4 – Number of recordings per subset

**What it does:** Counts real and fake recordings in the progress, eval and hidden subsets and compares the numbers with the main paper (p. 3, Table I). The paper does not give numbers for the hidden subset.

**Why:** Confirms that the dataset on disk is complete and the same as the one described in the paper.

In [ ]:
counts = pd.crosstab(cm["subset"], cm["key"]).reindex(SUBSETS)
counts["total"] = counts.sum(axis=1)
counts.loc["all"] = counts.sum()
display(counts)

expected_counts = {"progress": (1676, 14788), "eval": (14816, 133360), "hidden": None}
for subset, expected in expected_counts.items():
    measured = (int(counts.loc[subset, "bonafide"]), int(counts.loc[subset, "spoof"]))
    source = "main paper, p. 3, Table I" if expected else "not in the paper (observation)"
    add_check(f"{subset}: bona fide / spoof", source, expected, measured)

### Step 5 – Attacks per subset

**What it does:** Shows which fake-generating systems (attacks A07–A19) appear in each subset and how many recordings each has. Expected: all 13 attacks in every subset.

**Why:** Confirms that every subset contains the same kinds of fakes as ASVspoof 2019 LA eval.

In [ ]:
spoof = cm[cm["key"] == "spoof"]
attacks = pd.crosstab(spoof["attack"], spoof["subset"]).reindex(columns=SUBSETS, fill_value=0)
display(attacks)

expected_attacks = {f"A{i:02d}" for i in range(7, 20)}
for subset in SUBSETS:
    add_check(
        f"{subset}: attack IDs",
        "main paper, p. 3, section II-A",
        expected_attacks,
        set(spoof.loc[spoof["subset"] == subset, "attack"]),
    )

### Step 6 – Speakers

**What it does:** Counts the different speakers in each subset and compares the number with the main paper (p. 3, Table I: 37 women and 30 men in the progress and evaluation subsets). The label file has no information about gender, so gender is not checked.

**Why:** Confirms that all speakers of the dataset are present in the files.

In [ ]:
speakers = {subset: set(cm.loc[cm["subset"] == subset, "speaker_id"]) for subset in SUBSETS}
speaker_summary = pd.DataFrame(
    {"subset": SUBSETS, "speakers": [len(speakers[subset]) for subset in SUBSETS]}
).set_index("subset")
display(speaker_summary)

for subset in ["progress", "eval"]:
    add_check(
        f"{subset}: speakers",
        "main paper, p. 3, Table I (37 F + 30 M)",
        67,
        len(speakers[subset]),
    )

### Step 6b – Codecs and trimming

**What it does:** Counts the recordings for each codec (conditions LA-C1 to LA-C7) in each subset. Expected (main paper, p. 3, section II-A and Table II): the progress subset contains only C1–C4 (`none`, `alaw`, `pstn`, `g722`) and the eval subset all seven. It also checks that only the hidden subset contains recordings with the silence removed (main paper, p. 9, section IV-A).

**Why:** The codec and the bandwidth are two of the shortcuts studied in this project, and the hidden subset is the only part without silence. The values in the codec column are matched to the conditions by name – the files do not state this mapping.

In [ ]:
codec_subset = pd.crosstab(cm["codec"], cm["subset"]).reindex(
    index=CODECS, columns=SUBSETS, fill_value=0
)
display(codec_subset)

add_check(
    "progress: codecs (C1-C4)",
    "main paper, p. 3, section II-A and Table II",
    {"none", "alaw", "pstn", "g722"},
    set(cm.loc[cm["subset"] == "progress", "codec"]),
)
add_check(
    "eval: codecs (C1-C7)",
    "main paper, p. 3, section II-A and Table II",
    set(CODECS),
    set(cm.loc[cm["subset"] == "eval", "codec"]),
)
trim_subset = pd.crosstab(cm["trim"], cm["subset"]).reindex(columns=SUBSETS, fill_value=0)
display(trim_subset)

is_hidden = cm["subset"] == "hidden"
add_check(
    "hidden: all recordings trimmed (trim is not notrim)",
    "main paper, p. 9, section IV-A",
    True,
    bool((cm.loc[is_hidden, "trim"] != "notrim").all()),
)
add_check(
    "progress and eval: no trimmed recordings (trim is notrim)",
    "main paper, p. 9, section IV-A",
    True,
    bool((cm.loc[~is_hidden, "trim"] == "notrim").all()),
)

### Step 7 – Audio files vs labels

**What it does:** Compares the audio files in the `flac/` folder with the label file (labelled files missing on disk, audio files without a label) and checks that the list of evaluation files from the audio package (`ASVspoof2021.LA.cm.eval.trl.txt`) contains the same files, in the same order, as the label file.

**Why:** Every labelled recording must exist as a file, and every audio file must have a label. Unlike ASVspoof 2019, there are no extra recordings for the speaker verification task.

In [ ]:
on_disk = {p.stem for p in FLAC_DIR.glob("*.flac")}
in_labels = set(cm["file_name"])
missing = in_labels - on_disk
extra = on_disk - in_labels

print(f"Audio files on disk = {len(on_disk)}, rows in the label file = {len(in_labels)}")
add_check("Labelled files missing on disk", "-", 0, len(missing))
add_check("Audio files without a label", "-", 0, len(extra))
add_check(
    "Trial list = file names in trial_metadata.txt (same order)",
    "README.LA.txt, section 1 (list of evaluation data)",
    True,
    trials["file_name"].tolist() == cm["file_name"].tolist(),
)

file_summary = pd.DataFrame(
    [
        {
            "files_on_disk": len(on_disk),
            "rows_in_labels": len(in_labels),
            "rows_in_trial_list": len(trials),
            "missing_on_disk": len(missing),
            "without_label": len(extra),
        }
    ]
)
display(file_summary)

### Step 8a – Read the audio file properties

**What it does:** Reads only the header of every audio file (not the sound itself): format, sample rate, bit depth, number of channels and length. The result is saved to `outputs/02_asvspoof_2021_la_dataset_exploration_output/audio_info.csv`; the next time the notebook loads this file instead of reading all recordings again. **The first run takes a few minutes (181,566 files).**

**Why:** These properties are needed for the technical checks in the next cell and later for the table of recording features (silence, loudness, bandwidth, duration).

In [ ]:
INFO_CACHE = OUTPUTS_DIR / "audio_info.csv"

if INFO_CACHE.exists():
    info = pd.read_csv(INFO_CACHE, dtype={"file_name": str})
    print("Loaded from cache:", INFO_CACHE.relative_to(REPO_ROOT))
else:
    rows = []
    paths = sorted(FLAC_DIR.glob("*.flac"))
    print(f"Reading {len(paths)} files ...")
    for path in paths:
        meta = sf.info(path)
        rows.append(
            {
                "file_name": path.stem,
                "format": meta.format,
                "subtype": meta.subtype,
                "samplerate": meta.samplerate,
                "channels": meta.channels,
                "frames": meta.frames,
            }
        )
    info = pd.DataFrame(rows)
    info["duration_s"] = info["frames"] / info["samplerate"]
    info.to_csv(INFO_CACHE, index=False)
    print("Saved to:", INFO_CACHE.relative_to(REPO_ROOT))

audio = info.merge(cm, on="file_name", how="inner")
print("Audio files with labels:", len(audio))

### Step 8b – Check the audio file properties

**What it does:** Checks that all files are FLAC, 16 kHz and 16-bit (`README.LA.txt`, section 2; main paper, p. 2), shows the number of channels, and computes duration statistics for each subset, separately for real and fake recordings. The documentation gives no typical duration, so the median is only an observation.

**Why:** Confirms the technical description of the dataset. A clear difference in duration between real and fake recordings could be a shortcut that detectors use.

In [ ]:
add_check("File format", "README.LA.txt, section 2", {"FLAC"}, set(audio["format"]))
add_check(
    "Sample rate [Hz]",
    "README.LA.txt, section 2; main paper, p. 2",
    {16000},
    set(audio["samplerate"]),
)
add_check(
    "Bit depth (subtype)",
    "README.LA.txt, section 2; main paper, p. 2",
    {"PCM_16"},
    set(audio["subtype"]),
)
add_check("Number of channels", "not stated (observation)", None, set(audio["channels"]))

property_values = pd.DataFrame(
    {
        "property": ["format", "subtype", "samplerate", "channels"],
        "values found": [
            ", ".join(map(str, sorted(audio[col].unique())))
            for col in ["format", "subtype", "samplerate", "channels"]
        ],
    }
).set_index("property")
display(property_values)


duration_stats = audio.groupby(["subset", "key"])["duration_s"].describe()
duration_stats["count"] = duration_stats["count"].astype(int)
display(duration_stats.round(2))

median_duration = round(float(audio["duration_s"].median()), 2)
add_check("Median duration [s]", "not stated (observation)", None, median_duration)

### Step 8c – Duration histograms

**What it does:** Draws the same three charts as for ASVspoof 2019 LA (density, number of recordings, percent), with real and fake recordings on top of each other. Only the progress and eval subsets are used (original recordings, without the hidden subset).

**Why:** Makes it easy to see at a glance whether real and fake recordings differ in length.

In [ ]:
max_duration = float(np.ceil(audio["duration_s"].max()))
bins = np.linspace(0, max_duration, 101)


def plot_durations(data: pd.DataFrame, name: str) -> None:
    variants = [
        ("density", "Density", ""),
        ("counts", "Number of recordings", " (counts)"),
        ("percent", "Share of recordings in the group [%]", " (percent)"),
    ]
    for variant, y_label, title_end in variants:
        fig, ax = plt.subplots(figsize=(9, 4))
        for key, group in data.groupby("key"):
            weights = [100 / len(group)] * len(group) if variant == "percent" else None
            ax.hist(
                group["duration_s"],
                bins=bins,
                alpha=0.5,
                density=variant == "density",
                weights=weights,
                label=key,
            )
        ax.set_xlabel("Duration [s]")
        ax.set_ylabel(y_label)
        ax.set_title(f"ASVspoof 2021 LA, {name}: duration of bona fide vs spoof{title_end}")
        ax.legend()
        plt.show()


plot_durations(audio[audio["trim"] == "notrim"], "progress + eval")

### Step 9 – Summary of all checks

**What it does:** Shows all checks in one table with their status (OK, DIFF, INFO), lists only the differences and saves the table to `outputs/02_asvspoof_2021_la_dataset_exploration_output/checks.csv`.

**Why:** One place to see what matches the documentation and what does not.

In [ ]:
summary = pd.DataFrame(checks)
summary["status"] = summary["ok"].map({True: "OK", False: "DIFF"}).fillna("INFO")
summary = summary[["status", "check", "expected", "measured", "source"]]
display(summary)

print("\nDifferences:")
display(summary[summary["status"] == "DIFF"])

summary.to_csv(OUTPUTS_DIR / "checks.csv", index=False)
print("Saved to:", OUTPUTS_DIR.relative_to(REPO_ROOT) / "checks.csv")

### Step 10 – Report

**What it does:** Writes the results of all steps (tables only, no charts) to `results/02_asvspoof_2021_la/02_asvspoof_2021_la_dataset_exploration_output/dataset_exploration_report.md`.

**Why:** The results can be read later without running the notebook again.

In [ ]:
REPORT_PATH = RESULTS_DIR / "dataset_exploration_report.md"


def fmt(value, max_items: int = 10) -> str:
    if isinstance(value, set):
        if not value:
            return "{} (empty)"
        items = sorted(map(str, value))
        if len(items) > max_items:
            items = [*items[:max_items], f"... (+{len(value) - max_items} more)"]
        return "{" + ", ".join(items) + "}"
    if isinstance(value, float):
        return f"{value:.2f}"
    return str(value)


def df_to_md(df: pd.DataFrame, index: bool = True) -> str:
    data = df.reset_index() if index else df
    header = "| " + " | ".join(map(str, data.columns)) + " |"
    separator = "|" + "---|" * len(data.columns)
    rows = ["| " + " | ".join(fmt(v) for v in row) + " |" for row in data.itertuples(index=False)]
    return "\n".join([header, separator, *rows])


status_counts = summary["status"].value_counts()
differences = summary[summary["status"] == "DIFF"]

sections = [
    ("Step 4 – Number of recordings per subset", counts),
    ("Step 5 – Number of spoofed recordings per attack and subset", attacks),
    ("Step 6 – Number of speakers per subset", speaker_summary),
    ("Step 6b – Number of recordings per codec and subset", codec_subset),
    ("Step 6b – Number of recordings per trimming and subset", trim_subset),
    ("Step 7 – Audio files vs labels", file_summary),
    ("Step 8 – Audio file properties", property_values),
    ("Step 8 – Duration of recordings [s]", duration_stats.round(2)),
]

lines = [
    "# ASVspoof 2021 LA – dataset check report",
    "",
    f"- Generated: {datetime.now():%Y-%m-%d %H:%M}",
    f"- Dataset folder: `{DATA_DIR.relative_to(REPO_ROOT)}`",
    f"- Notebook: `notebooks/{NOTEBOOK_NAME}.ipynb`",
    "",
    "## Overview",
    "",
    f"- Checks in total: {len(summary)}",
    f"- OK (matches the documentation): {status_counts.get('OK', 0)}",
    f"- DIFF (does not match): {status_counts.get('DIFF', 0)}",
    f"- INFO (observation, nothing to compare with): {status_counts.get('INFO', 0)}",
    "",
    "## Differences",
    "",
    df_to_md(differences, index=False) if len(differences) else "No differences found.",
    "",
    "## All checks",
    "",
    df_to_md(summary, index=False),
    "",
]
for title, table in sections:
    lines += [f"## {title}", "", df_to_md(table), ""]

REPORT_PATH.write_text("\n".join(lines), encoding="utf-8")
print("Report saved to:", REPORT_PATH.relative_to(REPO_ROOT))